In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
df = pd.read_csv('final_gold.csv')

# Crear columnas con la media entre sensor 1 y sensor 2
df['avg_humidity_1_2'] = df[['sensor.sensor_temperatura_1_humidity', 'sensor.sensor_temperatura_2_humidity']].mean(axis=1)
df['avg_pressure_1_2'] = df[['sensor.sensor_temperatura_1_pressure', 'sensor.sensor_temperatura_2_pressure']].mean(axis=1)
df['avg_temperature_1_2'] = df[['sensor.sensor_temperatura_1_temperature', 'sensor.sensor_temperatura_2_temperature']].mean(axis=1)

# Entrenamos modelos separados para cada variable objetivo si hay datos suficientes
def train_and_predict(feature_col, target_col):
    train_data = df[[feature_col, target_col]].dropna()
    if train_data.empty:
        return None

    X_train = train_data[[feature_col]]
    y_train = train_data[target_col]

    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predecir valores faltantes
    missing_mask = df[target_col].isna() & df[feature_col].notna()
    df.loc[missing_mask, target_col] = model.predict(df.loc[missing_mask, [feature_col]]).round(2)

    return model

# Aplicar modelos
model_humidity = train_and_predict('avg_humidity_1_2', 'sensor.sensor_temperatura_3_humidity')
model_pressure = train_and_predict('avg_pressure_1_2', 'sensor.sensor_temperatura_3_pressure')
model_temperature = train_and_predict('avg_temperature_1_2', 'sensor.sensor_temperatura_3_temperature')

# Verificar cuántos valores se rellenaron
filled_counts = {
    'humidity_filled': df['sensor.sensor_temperatura_3_humidity'].notna().sum(),
    'pressure_filled': df['sensor.sensor_temperatura_3_pressure'].notna().sum(),
    'temperature_filled': df['sensor.sensor_temperatura_3_temperature'].notna().sum()
}

# Mostrar resultados
df_predicted = df[['time', 'sensor.sensor_temperatura_3_humidity',
                   'sensor.sensor_temperatura_3_pressure',
                   'sensor.sensor_temperatura_3_temperature']]





df.to_csv('final_silver_sensores_predichos.csv', index=False)

print("Predicción y guardado completados.")